In [73]:
from lark import Lark, ast_utils, Transformer, v_args
from typing import List
from dataclasses import dataclass, field
from lark.tree import Meta
import sys

In [74]:
this_module = sys.modules[__name__]

print(this_module)

<module '__main__'>


In [75]:
# source_code = '''
# let x = 5
# let y = 10

# def add(a, b):
#     return a + b
# end

# def func():
#     # TODO: Implement this function
# end


# let result = add(x, y)
# '''

In [ ]:
source_code = """


# Fibonacci sequence in Moose 

def fibo(n):
    let a = 0
    let b = 1
    
    while n >= 1:
        display(a)
        let temp = a
        let a = a + b
        let b = temp
        let n = n - 1
    end
end

let n = 10  # Number of Fibonacci terms to generate
fibo(n)

"""

In [77]:
grammar = r''' 
    start: code_block

    code_block: statement*

    ?statement: let_statement 
             | function_definition
             | return_statement
             | display_statement
             | if_statement
             | call_statement
             | whileloop_statement
             | break_statement

            

             
    # Statements

    let_statement: "let" NAME "=" expression

    function_definition: "def" NAME "(" [args] ")" ":" statement* "end"

    display_statement: "display" "(" expression? ")"

    if_statement: "if" expression ":" statement* ("else" ":" statement*)? "end"

    call_statement: NAME "(" [args] ")"

    return_statement: "return" expression

    args: expression ("," expression)*

    whileloop_statement: "while" expression ":" statement* "end"
    
    break_statement: "break"

    
 
    # Expressions

    expression: DEC_NUMBER -> number
        | STRING -> string
        | "true" -> boolean_true
        | "false" -> boolean_false
        | NAME "(" [args] ")" -> call_statement 
        | NAME -> name
        | expression OP expression -> binary_operation
        | expression COMPARISON_OP expression -> comparison_operation


    # Operators    
    OP: "+" | "-" | "*" | "/"

    COMPARISON_OP: "==" | "!=" | "<" | "<=" | ">" | ">="



    COMMENT: /#.*/
    NEWLINE: /(\r?\n)+/
    %ignore COMMENT
    %ignore NEWLINE
    %import python (NAME, DEC_NUMBER, STRING)
    %import common.WS
    %ignore WS

 
'''

In [78]:
def get_parser(grammar : str, parser_type: str = 'lalr', maybe_placeholders: bool = True):
    # Create and return a Lark parser instance using the specified grammar
    # Initialize the parser with the grammar and LALR parser algorithm
    return Lark(grammar=grammar, parser=parser_type, maybe_placeholders=maybe_placeholders)

In [79]:
# Test the parser with the source code
parsed_tree = get_parser(grammar).parse(source_code)
print(parsed_tree.pretty())

start
  code_block
    function_definition
      fibo
      args
        name	n
      let_statement
        a
        number	0
      let_statement
        b
        number	1
      whileloop_statement
        comparison_operation
          name	n
          >=
          number	1
        display_statement
          name	a
        let_statement
          temp
          name	a
        let_statement
          a
          binary_operation
            name	a
            +
            name	b
        let_statement
          b
          name	temp
        let_statement
          n
          binary_operation
            name	n
            -
            number	1
    let_statement
      n
      number	10
    call_statement
      fibo
      args
        name	n



In [80]:
# Transform the parse tree into a more usable format (Abstract Syntax Tree)


class _AST(ast_utils.Ast):
    # Base class for all AST nodes
    pass 

class _Statement(_AST):
    # Base class for all statement nodes
    pass

class _Expression(_AST):
    # Base class for all expression nodes
    pass

# AST node for a block of code containing multiple statements
@dataclass
class CodeBlock(_AST, ast_utils.AsList):
    statements: List[_Statement]

# AST node for a variable name
@dataclass
class Name(_Expression):
    name: str 

# AST node for a literal number
@dataclass
class Number(_Expression, ast_utils.WithMeta):
    meta: Meta
    value: int

# AST node for a literal string
@dataclass
class String(_Expression, ast_utils.WithMeta):
    meta: Meta
    value: str

# AST Node for a let statement
@dataclass
class LetStatement(_Statement):

    name: str
    value: _Expression

# AST Node for a boolean literal
@dataclass
class Boolean(_Expression, ast_utils.WithMeta):
    meta: Meta
    value: bool


# AST Node for print statement
@dataclass
class DisplayStatement(_Statement):
    value: _Expression

@dataclass
class IfStatement(_Statement):

    condition: _Expression
    body: List[_Statement]
    orelse: List[_Statement] = None

@dataclass
class FunctionDefinition(_Statement):
    name: str
    args: List[str] = field(default_factory=list)
    body: List[_Statement] = field(default_factory=list)

    def __init__(self, name: str, args=None, *body):
        self.name = name
        self.args = list(args) if args else []
        self.body = list(body)



@dataclass
class CallStatement(_Statement):

    name: str
    args: List[_Expression] = None

@dataclass
class ReturnStatement(_Statement):
    value: _Expression

@dataclass
class BinaryOperation(_Expression):
    left: _Expression
    op: str
    right: _Expression

@dataclass
class ComparisonOperation(_Expression):
    left: _Expression
    op: str
    right: _Expression


@dataclass
class WhileLoopStatement(_Statement):
    condition: _Expression
    body: List[_Statement]

@dataclass
class BreakStatement(_Statement):
    pass
    


@v_args(inline=True)
class ToAst(Transformer):
    
    def STRING(self, s):
    # Remove quotation marks
        return s[1:-1]

    def DEC_NUMBER(self, n):
        return int(n)
   
    def start(self, x):
        return x

    def boolean_true(self):
        return Boolean(meta=Meta(), value=True)

    def boolean_false(self):
        return Boolean(meta=Meta(), value=False)
    

    def statement(self, stmt):
        return stmt
    
    def if_statement(self, condition, body, orelse=None):
        return IfStatement(condition=condition, body=list(body), orelse=list(orelse) if orelse else None)

    def function_definition(self, name, args, *body):
        return FunctionDefinition(name=name, args=list(args) if args else [], body=list(body))

    def call_statement(self, name, args=None):
        return CallStatement(name=name, args=list(args) if args else None)


    def binary_operation(self, left, op, right):
        return BinaryOperation(left=left, op=op, right=right)

    def comparison_operation(self, left, op, right):
        return ComparisonOperation(left=left, op=op, right=right)



    def args(self, *body):
        return list(body)
        

    def whileloop_statement(self, condition, *body):
        return WhileLoopStatement(condition=condition, body=list(body))

    def break_statement(self):
        return BreakStatement()


In [81]:
transformer = ast_utils.create_transformer(this_module, ToAst())

In [82]:
def parse(source_code, parser: Lark) -> _AST:
    tree = parser.parse(source_code)
    return transformer.transform(tree)

In [83]:
parser = get_parser(grammar)
ast = parse(source_code, parser)
ast.statements

[FunctionDefinition(name=Token('NAME', 'fibo'), args=[Name(name=Token('NAME', 'n'))], body=[LetStatement(name=Token('NAME', 'a'), value=Number(meta=<lark.tree.Meta object at 0x000002A5BB9EFF20>, value=0)), LetStatement(name=Token('NAME', 'b'), value=Number(meta=<lark.tree.Meta object at 0x000002A5BB9EE7B0>, value=1)), WhileLoopStatement(condition=ComparisonOperation(left=Name(name=Token('NAME', 'n')), op=Token('COMPARISON_OP', '>='), right=Number(meta=<lark.tree.Meta object at 0x000002A5BB9EF840>, value=1)), body=[DisplayStatement(value=Name(name=Token('NAME', 'a'))), LetStatement(name=Token('NAME', 'temp'), value=Name(name=Token('NAME', 'a'))), LetStatement(name=Token('NAME', 'a'), value=BinaryOperation(left=Name(name=Token('NAME', 'a')), op=Token('OP', '+'), right=Name(name=Token('NAME', 'b')))), LetStatement(name=Token('NAME', 'b'), value=Name(name=Token('NAME', 'temp'))), LetStatement(name=Token('NAME', 'n'), value=BinaryOperation(left=Name(name=Token('NAME', 'n')), op=Token('OP', 

In [84]:
def transpile(node, indent=0):
    prefix = " " * indent

    if isinstance(node, CodeBlock):
        return "\n".join(
            transpile(statement, indent)
            for statement in node.statements
        )

    if isinstance(node, LetStatement):
        return f"{prefix}{str(node.name)} = {transpile(node.value)}"

    if isinstance(node, DisplayStatement):
        return f"{prefix}print({transpile(node.value)})"

    if isinstance(node, IfStatement):
        lines = [
            f"{prefix}if {transpile(node.condition)}:"
        ]
        if not hasattr(node.body, "__iter__"):
            lines.append(f"{transpile(node.body, indent + 4)}")
        else:
            lines.extend(
                transpile(statement, indent + 4)
                for statement in node.body 
            )

        if node.orelse:
            lines.append(f"{prefix}else:")
            if not hasattr(node.orelse, "__iter__"):
                lines.append(f"{transpile(node.orelse, indent + 4)}")
            else:
                lines.extend(
                    transpile(statement, indent + 4)
                    for statement in node.orelse
                )

        return "\n".join(lines)

    if isinstance(node, FunctionDefinition):
        lines = [
            f"def {node.name}({', '.join( transpile(arg) for arg in node.args) if node.args else ''}):"
        ]
        if not hasattr(node.body, "__iter__"):

            lines.append(f"{transpile(node.body, indent + 4)}")
        else:
            lines.extend(
                transpile(statement, indent + 4)
                for statement in node.body
            )
        return "\n".join(lines)

    if isinstance(node, CallStatement):
        return f"{node.name}({', '.join(transpile(arg) for arg in node.args) if node.args else ''})"


    if isinstance(node, Number):
        return str(node.value)

    if isinstance(node, String):
        return repr(node.value)

    if isinstance(node, Name):
        return str(node.name)

    if isinstance(node, Boolean):
        return True if node.value else False

    if isinstance(node, ReturnStatement):
        return f"{prefix}return {transpile(node.value)}"

    if isinstance(node, BinaryOperation):
        return f"{transpile(node.left)} {node.op} {transpile(node.right)}"

    if isinstance(node, WhileLoopStatement):
        lines = [
            f"{prefix}while {transpile(node.condition)}:"
        ]
        if not hasattr(node.body, "__iter__"):
            lines.append(f"{transpile(node.body, indent + 4)}")
        else:
            lines.extend(
                transpile(statement, indent + 4)
                for statement in node.body
            )
        return "\n".join(lines)

    if isinstance(node, BreakStatement):
        return f"{prefix}break"

    if isinstance(node, ComparisonOperation):
        return f"{transpile(node.left)} {node.op} {transpile(node.right)}"

    raise TypeError(f"Unsupported AST node: {type(node).__name__}")

In [85]:
source_code

'\n\n\n# Fibonacci sequence in Moose with while loop\n\ndef fibo(n):\n    let a = 0\n    let b = 1\n\n    while n >= 1:\n        display(a)\n        let temp = a\n        let a = a + b\n        let b = temp\n        let n = n - 1\n    end\nend\n\nlet n = 10  # Number of Fibonacci terms to generate\nfibo(n)\n\n'

In [86]:
parser = get_parser(grammar)
ast = parse(source_code, parser)
python_code = transpile(ast)
print(python_code)

def fibo(n):
    a = 0
    b = 1
    while n >= 1:
        print(a)
        temp = a
        a = a + b
        b = temp
        n = n - 1
n = 10
fibo(n)


In [87]:
# Execute python code as string
exec(python_code)

0
1
1
2
3
5
8
13
21
34


In [ ]:
# Fibonacci sequence in Python

def fibo(n):
    a,b = 0, 1

    while n >=1:
        print(a)
        a,b = a + b,a 
        n -= 1









In [106]:
n = 3  # Number of Fibonacci terms to generate
fibo(n)

0
1
1


In [ ]:
def execute_code(source_code):
    parser = get_parser(grammar)
    ast = parse(source_code, parser)
    python_code = transpile(ast)
    exec(python_code)

    

In [104]:
source_code = """
# Fibonacci sequence in Moose 

def fibo(n):
    let a = 0
    let b = 1
    
    while n >= 1:
        display(a)
        let temp = a
        let a = a + b
        let b = temp
        let n = n - 1
    end
end

let n = 3  # Number of Fibonacci terms to generate
fibo(n)
"""

In [105]:
execute_code(source_code)

0
1
1
